# Myeloid Cell Annotation Validation - Optimized Version

**Purpose**: Validate and refine myeloid annotations with robust marker coverage and visualization

**Key Optimizations**:
- Fixed marker coverage logic (var vs raw)
- Improved marker panels (pDC, neutrophil, interstitial macrophage)
- Robust dotplot/UMAP generation
- Comprehensive QC reports
- Vector format outputs (PDF + PNG)

**Critical Fixes**:
- P0-1: Dual marker coverage reporting (HVG vs full-gene)
- P0-2: Proper dotplot saving mechanism
- P0-3: UMAP existence check
- P1-1: pDC_Activated marker correction (remove plasma contamination)
- P1-2: Enhanced neutrophil panel (tissue-stable genes)
- P1-3: Resident IM marker axis

## 1. Configuration & Setup

In [ ]:
# ===== Import Libraries =====
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from scipy import sparse
warnings.filterwarnings('ignore')

# Print versions for reproducibility
print(f"Scanpy version: {sc.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("\nLibraries imported successfully")

In [ ]:
# ===== Configuration =====
# Paths
INPUT_PATH = "/home/h2048/data/py/0128/myeloid_analysis_unified/results/subcluster_unified_v2_20260128/adata_myeloid_subclustered_FINAL_v2_20260128.h5ad"
OUTPUT_DIR = Path("/home/h2048/data/py/0209/myeloid_validation_optimized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Column names
CELLTYPE_L3_COL = 'cell_type_L3'
CELLTYPE_L3_REFINED_COL = 'cell_type_L3_refined'
BATCH_KEY = 'sample'

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.n_jobs = 48
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(8, 6))

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Configuration complete")

## 2. Cell Type Re-annotation Mapping

In [ ]:
# ===== Cell Type Re-annotation Dictionary =====
CELLTYPE_REANNOTATION = {
    # Alveolar macrophage family
    'Alveolar macrophages_c0': 'Resident AM',
    'Alveolar macrophages_c1': 'Resident AM',
    'Alveolar macrophages_c2': 'Resident AM',
    'Alveolar macrophages_c3': 'Resting AM',
    
    # Neutrophils / Monocytes
    'Classical monocytes_c0': 'Neutrophils',
    'Classical monocytes_c1': 'Classical monocytes',
    'Classical monocytes_c2': 'Inflammatory classical monocytes',
    
    # DCs
    'DC2_c0': 'cDC2',
    'DC2_c1': 'Langerhans-like DC',
    'DC_c0': 'cDC2',
    'DC_c1': 'Mature antigen-presenting cDC2',
    
    # pDCs
    'pDC_c0': 'pDC',
    'pDC_c1': 'pDC',
    
    # Macrophages
    'Macrophages_c0': 'Inflammatory macrophages',
    'Macrophages_c1': 'M2-like interstitial macrophages',
    'Macrophages_c2': 'Atypical activated myeloid',
    
    # Mast cells
    'Mast cells_c0': 'Mast cells',
    'Mast cells_c1': 'Mast cells',
    
    # Intestinal → Interstitial macrophages
    'Intestinal macrophages_c0': 'Interstitial macrophages (M2-like CD163L1+)',
    'Intestinal macrophages_c1': 'Interstitial macrophages (low-quality)',
    'Intestinal macrophages_c2': 'Interstitial macrophages (tissue-remodeling)',
    'Intestinal macrophages_c3': 'Interstitial macrophages (immunoregulatory)',
    'Intestinal macrophages_c4': 'Interstitial macrophages (stromal-like)',
}

LOW_CONFIDENCE_CLUSTERS = [
    'Macrophages_c2',
    'Intestinal macrophages_c1',
    'Intestinal macrophages_c3',
    'Intestinal macrophages_c4',
]

print(f"Cell type re-annotation defined: {len(CELLTYPE_REANNOTATION)} mappings")
print(f"Low-confidence clusters: {len(LOW_CONFIDENCE_CLUSTERS)}")

## 3. Optimized Marker Gene Sets

**Key Improvements**:
- P1-1: pDC_Activated now uses IFN/ISG markers (no plasma contamination)
- P1-2: Neutrophil panel includes tissue-stable genes
- P1-3: Added Resident_IM axis for interstitial macrophages

In [ ]:
# ===== Subtype-Specific Markers (Optimized) =====
SUBTYPE_MARKERS = {
    # Alveolar Macrophages
    'AM_General': ['MARCO', 'PPARG', 'FABP4', 'LPL', 'APOE', 'CSF2RA', 'ITGAX'],
    'AM_Lipid_Handling': ['FABP4', 'LPL', 'APOE', 'APOC1', 'TREM2', 'CD36', 'HILPDA', 'SCD'],
    'AM_Surfactant': ['SFTPA1', 'SFTPA2', 'SFTPB', 'SFTPC', 'SFTPD'],
    'AM_Resting': ['CD5L', 'FABP4', 'CHI3L1'],
    
    # Monocytes
    'Classical_Mono': ['CD14', 'S100A8', 'S100A9', 'S100A12', 'FCN1', 'VCAN', 'LGALS3'],
    'Inflammatory_Mono': ['NLRP3', 'IL1B', 'CXCL8', 'CCL3', 'CCL4', 'S100A8', 'S100A9'],
    'Non_Classical_Mono': ['FCGR3A', 'CX3CR1', 'LST1', 'MS4A7', 'IFITM3'],
    
    # Neutrophils (P1-2: Enhanced with tissue-stable genes)
    'Neutrophil_Core': ['CXCR1', 'CXCR2', 'FCGR3B', 'CSF3R'],
    'Neutrophil_Granule': ['MPO', 'ELANE', 'AZU1', 'DEFA3', 'DEFA4'],
    'Neutrophil_Tissue_Stable': ['S100A8', 'S100A9', 'S100A12', 'LTF', 'LCN2', 'MMP8'],
    
    # DCs
    'cDC2_General': ['CD1C', 'FCER1A', 'CLEC10A', 'CD1E', 'IRF4'],
    'cDC2_Mature': ['CIITA', 'RFX5', 'CCL17', 'ALCAM', 'CD40', 'CD86'],
    'cDC1': ['XCR1', 'CLEC9A', 'BATF3', 'IRF8', 'CADM1'],
    'pDC_Core': ['IL3RA', 'CLEC4C', 'TCF4', 'LILRA4'],
    'pDC_IFN_Activated': ['IRF7', 'ISG15', 'IFIT1', 'IFIT3', 'MX1', 'OAS1', 'OASL', 'RSAD2', 'STAT1'],
    'Langerhans_DC': ['CD207', 'CD1A', 'FCER1A'],
    'Mature_DC_Migration': ['CCR7', 'LAMP3', 'FSCN1', 'RELB', 'CCL17', 'CCL22'],
    
    # Macrophages (General)
    'Mac_Inflammatory': ['IL1B', 'PTX3', 'CCL20', 'CXCL8', 'TNF', 'NFKBIA'],
    'Mac_M2_Interstitial': ['STAB1', 'F13A1', 'CLEC10A', 'MRC1', 'CD163'],
    'Mac_Resident_IM': ['LYVE1', 'FOLR2', 'MERTK', 'MARCKS', 'GAS6', 'MS4A7', 'C1QC'],  # P1-3: Added!
    'Mac_Tissue_Remodeling': ['MMP9', 'MMP12', 'TIMP1', 'FN1', 'SPP1'],
    'Mac_Immunoregulatory': ['TGFB1', 'IL10', 'VEGFA', 'CD163'],
    
    # ⭐ Intestinal/Interstitial Macrophages Specific (保留用于可视化)
    'Intestinal_Mac_Niche': ['CD163L1', 'SELENOP', 'ABCA6', 'PLXDC1', 'CXCL12'],  # Original niche markers
    'Intestinal_Mac_Stromal_Like': ['PLXDC1', 'CXCL12', 'PDGFRA', 'DCN', 'LUM'],  # Stromal contamination markers
    'Intestinal_Mac_Core': ['CD163', 'MRC1', 'LYVE1', 'C1QA', 'C1QB', 'C1QC'],  # Core intestinal Mac
    
    # Mast cells
    'Mast': ['KIT', 'CPA3', 'TPSAB1', 'TPSB2', 'MS4A2', 'HDC', 'HPGDS'],
    
    # Core myeloid
    'Pan_Myeloid': ['LYZ', 'LST1', 'TYROBP', 'FCER1G', 'AIF1', 'SPI1'],
    'APC_Core': ['HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1', 'CD74'],
    'Complement': ['C1QA', 'C1QB', 'C1QC', 'APOE'],
}

# ===== Functional State Markers =====
STATE_MARKERS = {
    'Type_I_IFN': ['ISG15', 'IFIT1', 'IFIT3', 'MX1', 'OAS1', 'OASL', 'RSAD2', 'IRF7'],
    'IL1_NFKB_Inflammation': ['IL1B', 'NLRP3', 'TNF', 'NFKBIA', 'CXCL2', 'CCL3', 'CCL4'],
    'Hypoxia_Glycolysis': ['SLC2A1', 'LDHA', 'HK2', 'PFKFB3', 'HILPDA'],
    'Proliferation': ['MKI67', 'TOP2A', 'STMN1', 'HMGB2', 'PCNA'],
    'Chemokine_Production': ['CCL2', 'CCL3', 'CCL4', 'CXCL2', 'CXCL8', 'CXCL10'],
    'Lipid_Metabolism': ['PPARG', 'FABP4', 'LPL', 'APOE', 'CD36', 'TREM2'],
}

# ===== Contamination Markers =====
CONTAMINATION_MARKERS = {
    'Epithelial': ['EPCAM', 'KRT19', 'KRT18', 'KRT8', 'CDH1'],
    'AT2_Alveolar': ['SFTPA1', 'SFTPA2', 'SFTPB', 'SFTPC', 'SFTPD', 'NKX2-1'],
    'Airway_Epithelial': ['BPIFA1', 'SCGB1A1', 'MUC5AC', 'FOXJ1'],
    'T_Cell': ['CD3D', 'CD3E', 'CD4', 'CD8A', 'IL7R'],
    'B_Cell': ['MS4A1', 'CD79A', 'CD79B', 'IGHM', 'IGKC'],
    'Plasma_Cell': ['MZB1', 'XBP1', 'JCHAIN', 'SDC1'],
    'NK_Cell': ['NKG7', 'GNLY', 'PRF1', 'GZMB', 'NCAM1'],
    'Endothelial': ['PECAM1', 'VWF', 'KDR', 'EMCN', 'CDH5'],
    'Lymphatic_Endo': ['PROX1', 'PDPN', 'LYVE1', 'CCL21'],
    'Fibroblast': ['COL1A1', 'COL3A1', 'DCN', 'LUM'],
    'Pericyte_SMC': ['RGS5', 'PDGFRB', 'ACTA2', 'TAGLN', 'MYH11'],
    'RBC': ['HBB', 'HBA1', 'HBA2', 'ALAS2'],
    'Platelet': ['PPBP', 'PF4', 'GP9'],
}

print(f"\nOptimized marker sets defined:")
print(f"  Subtype markers: {len(SUBTYPE_MARKERS)} categories")
print(f"  State markers: {len(STATE_MARKERS)} categories")
print(f"  Contamination markers: {len(CONTAMINATION_MARKERS)} categories")
print(f"\nKey improvements:")
print(f"  ✓ pDC_IFN_Activated: Now uses ISG/IFN markers (no plasma contamination)")
print(f"  ✓ Neutrophil_Tissue_Stable: Added tissue-resident stable genes")
print(f"  ✓ Mac_Resident_IM: Added resident interstitial macrophage axis")

## 4. Load Data & Validate Structure

**P0-3 Fix**: Check UMAP existence before visualization

In [ ]:
# ===== Load Dataset =====
print("Loading myeloid dataset...")
adata = sc.read_h5ad(INPUT_PATH)

print(f"Dataset loaded: {adata.n_obs:,} cells × {adata.n_vars} genes")
print(f"\nData structure:")
print(f"  Layers: {list(adata.layers.keys())}")
print(f"  Obsm: {list(adata.obsm.keys())}")
print(f"  .raw present: {adata.raw is not None}")
if adata.raw is not None:
    print(f"  .raw genes: {adata.raw.n_vars}")

# P0-3: CRITICAL CHECK
if 'X_umap' not in adata.obsm:
    raise ValueError(
        "❌ X_umap not found in adata.obsm!\n"
        "   Please compute UMAP first: sc.tl.umap(adata)"
    )
print(f"  ✓ X_umap available: {adata.obsm['X_umap'].shape}")

In [ ]:
# ===== Standardize L3 Labels =====
def standardize_l3_labels(adata, l2_key='cell_type_L2', l3_key='cell_type_L3'):
    """Standardize L3 labels to hierarchical format"""
    if l3_key not in adata.obs.columns:
        print(f"  ⚠️  {l3_key} not found")
        return adata
    
    sample = str(adata.obs[l3_key].iloc[0])
    if '_c' in sample:
        print(f"  ✓ {l3_key} already hierarchical")
        return adata
    
    print(f"\nStandardizing L3 labels...")
    adata.obs['cell_type_L3_original'] = adata.obs[l3_key].copy()
    adata.obs['subcluster_id'] = pd.to_numeric(
        adata.obs[l3_key].astype(str), errors='coerce'
    ).astype('Int64')
    
    # CRITICAL FIX: Convert categorical to object first
    if isinstance(adata.obs[l3_key].dtype, pd.CategoricalDtype):
        adata.obs[l3_key] = adata.obs[l3_key].astype('object')
    
    l2 = adata.obs[l2_key].astype(str)
    sid = adata.obs['subcluster_id']
    mask = l2.notna() & sid.notna()
    adata.obs.loc[mask, l3_key] = (
        l2[mask] + '_c' + sid[mask].astype('Int64').astype(str)
    )
    
    tmp_df = adata.obs[[l2_key, 'subcluster_id', l3_key]].dropna()
    tmp_df = tmp_df.sort_values([l2_key, 'subcluster_id'])
    ordered_levels = pd.unique(tmp_df[l3_key].astype(str)).tolist()
    
    adata.obs[l3_key] = pd.Categorical(
        adata.obs[l3_key].astype(str),
        categories=ordered_levels,
        ordered=True
    )
    
    print(f"  ✓ Standardized {len(ordered_levels)} L3 subclusters")
    return adata

adata = standardize_l3_labels(adata, l3_key=CELLTYPE_L3_COL)

## 5. Apply Re-annotation

In [ ]:
# ===== Apply Re-annotation =====
print("\nApplying cell type re-annotation...\n")

adata.obs[CELLTYPE_L3_REFINED_COL] = adata.obs[CELLTYPE_L3_COL].map(CELLTYPE_REANNOTATION)
unmapped = adata.obs[CELLTYPE_L3_COL][adata.obs[CELLTYPE_L3_REFINED_COL].isna()].unique()

if len(unmapped) > 0:
    print(f"⚠️  {len(unmapped)} unmapped clusters:")
    for ct in unmapped:
        print(f"    {ct}: {(adata.obs[CELLTYPE_L3_COL] == ct).sum():,} cells")
    adata.obs[CELLTYPE_L3_REFINED_COL].fillna(adata.obs[CELLTYPE_L3_COL], inplace=True)

adata.obs[CELLTYPE_L3_REFINED_COL] = adata.obs[CELLTYPE_L3_REFINED_COL].astype('category')

print(f"\n✓ Re-annotation complete")
print(f"  Unique refined labels: {adata.obs[CELLTYPE_L3_REFINED_COL].nunique()}")

# Display
print("\n" + "="*80)
print("REFINED CELL TYPE ANNOTATIONS")
print("="*80)
refined_counts = adata.obs[CELLTYPE_L3_REFINED_COL].value_counts()
for ct, count in refined_counts.items():
    pct = count / adata.n_obs * 100
    original = adata.obs[adata.obs[CELLTYPE_L3_REFINED_COL] == ct][CELLTYPE_L3_COL].iloc[0]
    flag = " [⚠️ QC]" if original in LOW_CONFIDENCE_CLUSTERS else ""
    print(f"  {ct}: {count:,} cells ({pct:.1f}%){flag}")
print("="*80)

## 6. Prepare Data & Dual Marker Coverage Report

**P0-1 Fix**: Separate coverage reporting for var (HVG) vs raw (full-gene)

In [ ]:
# ===== Data Preparation =====
print("Preparing data for analysis...")

# Ensure log1p layer
if 'log1p' in adata.layers:
    print("  ✓ Using existing log1p layer")
    viz_layer = 'log1p'
elif 'counts' in adata.layers:
    print("  ⚙️  Generating log1p layer from counts")
    adata.layers['log1p'] = adata.layers['counts'].copy()
    sc.pp.normalize_total(adata, target_sum=1e4, layer='log1p')
    sc.pp.log1p(adata, layer='log1p')
    viz_layer = 'log1p'
else:
    print("  ⚠️  No counts/log1p layer, using .X")
    viz_layer = None

# P0-1: Determine gene source strategy
has_raw = adata.raw is not None
var_genes = set(adata.var_names)
raw_genes = set(adata.raw.var_names) if has_raw else set()

print(f"\n  Gene availability:")
print(f"    .var_names (HVG subset): {len(var_genes)}")
if has_raw:
    print(f"    .raw.var_names (full-gene): {len(raw_genes)}")
    print(f"    Genes only in .raw: {len(raw_genes - var_genes)}")

# P0-1: For visualization, prefer .raw if available (better marker coverage)
use_raw_for_viz = has_raw
print(f"\n  Visualization strategy:")
print(f"    use_raw: {use_raw_for_viz}")
print(f"    layer: {viz_layer if not use_raw_for_viz else 'N/A (using .raw)'}")

In [ ]:
# ===== P0-1: Dual Marker Coverage Report =====
print("\nGenerating dual marker coverage report...")

def check_marker_coverage_dual(marker_dict, var_genes, raw_genes):
    """Check coverage in both var (HVG) and raw (full-gene)"""
    results = []
    for category, genes in marker_dict.items():
        present_var = [g for g in genes if g in var_genes]
        present_raw = [g for g in genes if g in raw_genes]
        missing = [g for g in genes if g not in raw_genes]
        
        results.append({
            'category': category,
            'n_total': len(genes),
            'n_in_var': len(present_var),
            'n_in_raw': len(present_raw),
            'coverage_var_pct': len(present_var) / len(genes) * 100 if genes else 0,
            'coverage_raw_pct': len(present_raw) / len(genes) * 100 if genes else 0,
            'missing_genes': ', '.join(missing) if missing else 'None',
        })
    return pd.DataFrame(results)

# Run coverage analysis
coverage_subtype = check_marker_coverage_dual(SUBTYPE_MARKERS, var_genes, raw_genes)
coverage_state = check_marker_coverage_dual(STATE_MARKERS, var_genes, raw_genes)
coverage_contam = check_marker_coverage_dual(CONTAMINATION_MARKERS, var_genes, raw_genes)

# Combine and save
coverage_subtype['marker_type'] = 'Subtype'
coverage_state['marker_type'] = 'State'
coverage_contam['marker_type'] = 'Contamination'

coverage_all = pd.concat([coverage_subtype, coverage_state, coverage_contam], ignore_index=True)
coverage_all = coverage_all.sort_values('coverage_raw_pct', ascending=False)

# Save detailed report
coverage_all.to_csv(OUTPUT_DIR / 'marker_coverage_dual_report.csv', index=False)
print(f"  ✓ Saved: marker_coverage_dual_report.csv")

# Display summary
print("\n" + "="*80)
print("MARKER COVERAGE SUMMARY")
print("="*80)

low_coverage = coverage_all[
    (coverage_all['coverage_var_pct'] < 50) | 
    (coverage_all['coverage_raw_pct'] < 50)
]

if len(low_coverage) > 0:
    print("\nCategories with <50% coverage in either var or raw:")
    print("-" * 80)
    for _, row in low_coverage.iterrows():
        print(f"\n  {row['category']} ({row['marker_type']})")
        print(f"    In .var (HVG): {row['n_in_var']}/{row['n_total']} ({row['coverage_var_pct']:.0f}%)")
        print(f"    In .raw (full): {row['n_in_raw']}/{row['n_total']} ({row['coverage_raw_pct']:.0f}%)")
        if row['missing_genes'] != 'None':
            print(f"    Missing: {row['missing_genes']}")
else:
    print("\n✓ All marker categories have >50% coverage in both var and raw")

print("\n" + "="*80)

## 7. Signature Scoring (with reproducibility)

In [ ]:
# ===== Signature Scoring =====
print("Computing signature scores...\n")

def get_present_genes(genes, available_genes):
    """Get genes present in dataset"""
    return [g for g in genes if g in available_genes]

# Use raw genes for better coverage
scoring_genes = raw_genes if has_raw else var_genes

# Combine all signatures
ALL_SIGNATURES = {}
for cat, genes in SUBTYPE_MARKERS.items():
    ALL_SIGNATURES[f'SUB_{cat}'] = genes
for cat, genes in STATE_MARKERS.items():
    ALL_SIGNATURES[f'STATE_{cat}'] = genes
for cat, genes in CONTAMINATION_MARKERS.items():
    ALL_SIGNATURES[f'CONTAM_{cat}'] = genes

# Compute scores
computed = []
skipped = []

for sig_name, genes in ALL_SIGNATURES.items():
    present = get_present_genes(genes, scoring_genes)
    
    if len(present) < 3:
        skipped.append((sig_name, len(present)))
        continue
    
    try:
        # P2-2: Fixed random state for reproducibility
        sc.tl.score_genes(
            adata,
            gene_list=present,
            score_name=f'score_{sig_name}',
            use_raw=use_raw_for_viz,
            layer=None if use_raw_for_viz else viz_layer,
            random_state=RANDOM_SEED,
            ctrl_size=50,
        )
        computed.append(sig_name)
    except Exception as e:
        print(f"  ⚠️  Failed: {sig_name} - {e}")
        skipped.append((sig_name, len(present)))

score_cols = [col for col in adata.obs.columns if col.startswith('score_')]

print(f"Signature scoring complete:")
print(f"  ✓ Computed: {len(computed)} signatures")
print(f"  ⚠️  Skipped: {len(skipped)} signatures")
print(f"  Total score columns: {len(score_cols)}")

## 8. Visualization Part 1: UMAPs

In [ ]:
# ===== UMAP: Annotations =====
print("Generating UMAP visualizations...\n")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sc.pl.umap(adata, color=CELLTYPE_L3_COL, ax=axes[0],
           title='Original L3 Annotations', legend_loc='right margin',
           legend_fontsize=6, show=False)

sc.pl.umap(adata, color=CELLTYPE_L3_REFINED_COL, ax=axes[1],
           title='Refined Cell Type Annotations', legend_loc='right margin',
           legend_fontsize=6, show=False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_original_vs_refined.png', dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'umap_original_vs_refined.pdf', bbox_inches='tight')  # P2-4: Vector format
plt.show()

print("  ✓ Saved: umap_original_vs_refined (PNG + PDF)")

In [ ]:
# ===== UMAP: Key Markers =====
print("\nGenerating key marker UMAPs...")

KEY_MARKERS = {
    'AM': ['MARCO', 'FABP4', 'PPARG', 'SFTPC'],
    'Monocytes': ['CD14', 'S100A8', 'FCGR3A', 'CX3CR1'],
    'Neutrophils': ['CXCR1', 'CXCR2', 'FCGR3B', 'S100A12'],
    'DCs': ['CD1C', 'FCER1A', 'CLEC9A', 'IL3RA', 'IRF7'],
    'Macrophages': ['CD163L1', 'SELENOP', 'IL1B', 'STAB1', 'LYVE1'],
    'Mast': ['KIT', 'CPA3', 'TPSAB1'],
}

marker_genes_for_viz = raw_genes if use_raw_for_viz else var_genes

for category, markers in KEY_MARKERS.items():
    available = [m for m in markers if m in marker_genes_for_viz]
    
    if len(available) == 0:
        print(f"  ⚠️  No markers available for {category}")
        continue
    
    n = len(available)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for idx, marker in enumerate(available):
        try:
            sc.pl.umap(adata, color=marker, ax=axes[idx], title=marker,
                      vmax='p99', show=False, use_raw=use_raw_for_viz,
                      layer=None if use_raw_for_viz else viz_layer)
        except Exception as e:
            print(f"  ⚠️  Failed to plot {marker}: {e}")
            axes[idx].axis('off')
    
    for idx in range(len(available), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f'{category} Marker Expression', fontsize=16, y=1.00)
    plt.tight_layout()
    
    filename_base = f'umap_markers_{category.lower().replace(" ", "_")}'
    plt.savefig(OUTPUT_DIR / f'{filename_base}.png', dpi=300, bbox_inches='tight')
    plt.savefig(OUTPUT_DIR / f'{filename_base}.pdf', bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ Saved: {filename_base} ({len(available)} markers)")

print("\n✓ Key marker UMAPs complete")

## 9. Visualization Part 2: Dotplots (P0-2 Fixed)

**Critical Fix**: Proper dotplot saving mechanism

In [ ]:
# ===== P0-2: Fixed Dotplot Function (Robust Version) =====
def plot_marker_dotplot_fixed(adata, markers, groupby, title, filename,
                               use_raw=False, layer=None, figsize=(12, 8)):
    """
    Generate marker dotplot with proper saving (P0-2 fix + robust error handling)
    """
    gene_names = adata.raw.var_names if use_raw else adata.var_names
    available_markers = [m for m in markers if m in gene_names]
    
    if len(available_markers) == 0:
        print(f"  ⚠️  No markers for {title}")
        return False
    
    try:
        # Attempt 1: Modern scanpy with return_fig
        try:
            dp = sc.pl.dotplot(
                adata,
                var_names=available_markers,
                groupby=groupby,
                use_raw=use_raw,
                layer=None if use_raw else layer,
                standard_scale='var',
                show=False,
                return_fig=True,
            )
            if dp is not None and hasattr(dp, 'fig'):
                dp.fig.suptitle(title, fontsize=14, y=1.02)
                dp.fig.set_size_inches(figsize)
                dp.fig.savefig(OUTPUT_DIR / f'{filename}.png', dpi=300, bbox_inches='tight')
                dp.fig.savefig(OUTPUT_DIR / f'{filename}.pdf', bbox_inches='tight')
                plt.close(dp.fig)
                print(f"  ✓ Saved: {filename} ({len(available_markers)} markers)")
                return True
        except (TypeError, AttributeError):
            pass
        
        # Attempt 2: Use DotPlot object's savefig method
        try:
            dp = sc.pl.dotplot(
                adata,
                var_names=available_markers,
                groupby=groupby,
                use_raw=use_raw,
                layer=None if use_raw else layer,
                standard_scale='var',
                show=False,
            )
            if dp is not None and hasattr(dp, 'savefig'):
                dp.savefig(OUTPUT_DIR / f'{filename}.png', dpi=300)
                dp.savefig(OUTPUT_DIR / f'{filename}.pdf')
                print(f"  ✓ Saved: {filename} ({len(available_markers)} markers)")
                return True
        except (TypeError, AttributeError):
            pass
        
        # Attempt 3: Create figure explicitly and use make_grid_spec
        fig = plt.figure(figsize=figsize)
        ax_dict = sc.pl.dotplot(
            adata,
            var_names=available_markers,
            groupby=groupby,
            use_raw=use_raw,
            layer=None if use_raw else layer,
            standard_scale='var',
            show=False,
            ax=fig.add_subplot(111) if hasattr(fig, 'add_subplot') else None,
        )
        
        if plt.gcf().get_axes():  # Check if plot was created
            plt.suptitle(title, fontsize=14, y=1.02)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / f'{filename}.png', dpi=300, bbox_inches='tight')
            plt.savefig(OUTPUT_DIR / f'{filename}.pdf', bbox_inches='tight')
            plt.close()
            print(f"  ✓ Saved: {filename} ({len(available_markers)} markers)")
            return True
        
        print(f"  ⚠️  Could not create plot for {filename}")
        return False
        
    except Exception as e:
        print(f"  ❌ Failed: {filename} - {e}")
        plt.close('all')  # Clean up any open figures
        return False

print("\nGenerating marker dotplots (with P0-2 fix)...")
print("=" * 80)

In [ ]:
# ===== Dotplot 1: Core Myeloid =====
core_markers = (
    SUBTYPE_MARKERS['Pan_Myeloid'] +
    SUBTYPE_MARKERS['APC_Core'] +
    SUBTYPE_MARKERS['Complement']
)
plot_marker_dotplot_fixed(
    adata, core_markers, CELLTYPE_L3_REFINED_COL,
    'Core Myeloid & Antigen Presentation Markers',
    'dotplot_core_myeloid',
    use_raw=use_raw_for_viz, layer=viz_layer, figsize=(14, 10)
)

In [ ]:
# ===== Dotplot 2: Alveolar Macrophages =====
am_markers = (
    SUBTYPE_MARKERS['AM_General'] +
    SUBTYPE_MARKERS['AM_Lipid_Handling'] +
    SUBTYPE_MARKERS['AM_Surfactant'] +
    SUBTYPE_MARKERS['AM_Resting']
)
plot_marker_dotplot_fixed(
    adata, am_markers, CELLTYPE_L3_REFINED_COL,
    'Alveolar Macrophage Markers',
    'dotplot_alveolar_macrophages',
    use_raw=use_raw_for_viz, layer=viz_layer, figsize=(16, 10)
)

In [ ]:
# ===== Dotplot 3: Monocytes & Neutrophils (Enhanced) =====
mono_neut_markers = (
    SUBTYPE_MARKERS['Classical_Mono'] +
    SUBTYPE_MARKERS['Inflammatory_Mono'] +
    SUBTYPE_MARKERS['Non_Classical_Mono'] +
    SUBTYPE_MARKERS['Neutrophil_Core'] +
    SUBTYPE_MARKERS['Neutrophil_Tissue_Stable']  # P1-2: Added!
)
plot_marker_dotplot_fixed(
    adata, mono_neut_markers, CELLTYPE_L3_REFINED_COL,
    'Monocyte & Neutrophil Markers (Enhanced)',
    'dotplot_monocytes_neutrophils',
    use_raw=use_raw_for_viz, layer=viz_layer, figsize=(18, 10)
)

In [ ]:
# ===== Dotplot 4: Dendritic Cells (Fixed pDC) =====
dc_markers = (
    SUBTYPE_MARKERS['cDC2_General'] +
    SUBTYPE_MARKERS['cDC2_Mature'] +
    SUBTYPE_MARKERS['cDC1'] +
    SUBTYPE_MARKERS['pDC_Core'] +
    SUBTYPE_MARKERS['pDC_IFN_Activated'] +  # P1-1: Fixed!
    SUBTYPE_MARKERS['Langerhans_DC'] +
    SUBTYPE_MARKERS['Mature_DC_Migration']
)
plot_marker_dotplot_fixed(
    adata, dc_markers, CELLTYPE_L3_REFINED_COL,
    'Dendritic Cell Markers (pDC Fixed)',
    'dotplot_dendritic_cells',
    use_raw=use_raw_for_viz, layer=viz_layer, figsize=(20, 10)
)

In [ ]:
# ===== Dotplot 5: Interstitial Macrophages (Enhanced + Original Intestinal Mac Markers) =====
mac_markers = (
    SUBTYPE_MARKERS['Mac_Inflammatory'] +
    SUBTYPE_MARKERS['Mac_M2_Interstitial'] +
    SUBTYPE_MARKERS['Mac_Resident_IM'] +  # P1-3: Added!
    SUBTYPE_MARKERS['Mac_Tissue_Remodeling'] +
    SUBTYPE_MARKERS['Mac_Immunoregulatory'] +
    SUBTYPE_MARKERS['Intestinal_Mac_Niche'] +  # ⭐ 原始肠道巨噬细胞niche markers
    SUBTYPE_MARKERS['Intestinal_Mac_Stromal_Like'] +  # ⭐ 基质样marker
    SUBTYPE_MARKERS['Intestinal_Mac_Core']  # ⭐ 核心肠道巨噬细胞marker
)
plot_marker_dotplot_fixed(
    adata, mac_markers, CELLTYPE_L3_REFINED_COL,
    'Interstitial & Inflammatory Macrophage Markers (Enhanced + Original Intestinal Mac)',
    'dotplot_interstitial_macrophages',
    use_raw=use_raw_for_viz, layer=viz_layer, figsize=(20, 12)  # 加大尺寸
)

In [ ]:
# ===== Dotplot 6-8: Mast, States, Contamination =====
plot_marker_dotplot_fixed(
    adata, SUBTYPE_MARKERS['Mast'], CELLTYPE_L3_REFINED_COL,
    'Mast Cell Markers', 'dotplot_mast_cells',
    use_raw=use_raw_for_viz, layer=viz_layer, figsize=(12, 8)
)

state_markers = []
for m in STATE_MARKERS.values():
    state_markers.extend(m)
plot_marker_dotplot_fixed(
    adata, state_markers, CELLTYPE_L3_REFINED_COL,
    'Functional State Markers', 'dotplot_functional_states',
    use_raw=use_raw_for_viz, layer=viz_layer, figsize=(16, 10)
)

contam_markers = []
for m in CONTAMINATION_MARKERS.values():
    contam_markers.extend(m)
plot_marker_dotplot_fixed(
    adata, contam_markers, CELLTYPE_L3_REFINED_COL,
    'Contamination Markers', 'dotplot_contamination',
    use_raw=use_raw_for_viz, layer=viz_layer, figsize=(18, 10)
)

print("\n" + "="*80)
print("✓ Dotplot generation complete")
print("="*80)

In [ ]:
# ===== Extract Dotplot Data to CSV (Long Format) =====
print("\nExtracting dotplot data to CSV (long format)...")
print("=" * 80)

def calculate_dotplot_stats(adata, markers, groupby, use_raw=False, layer=None):
    """
    Calculate mean expression and percent expressed for dotplot
    Returns long-format DataFrame
    """
    gene_names = adata.raw.var_names if use_raw else adata.var_names
    available_markers = [m for m in markers if m in gene_names]
    
    if len(available_markers) == 0:
        return None
    
    results = []
    
    for cluster in adata.obs[groupby].unique():
        mask = adata.obs[groupby] == cluster
        n_cells = mask.sum()
        
        for gene in available_markers:
            # Get expression data
            if use_raw:
                expr = adata.raw[mask, gene].X
            else:
                if layer is not None and layer in adata.layers:
                    expr = adata[mask, gene].layers[layer]
                else:
                    expr = adata[mask, gene].X
            
            # Convert to array if sparse
            if sparse.issparse(expr):
                expr = expr.toarray().flatten()
            else:
                expr = np.asarray(expr).flatten()
            
            # Calculate statistics
            mean_expr = np.mean(expr)
            pct_expressed = (expr > 0).sum() / len(expr) * 100 if len(expr) > 0 else 0
            
            results.append({
                'cluster': str(cluster),
                'gene': gene,
                'mean_expression': mean_expr,
                'pct_expressed': pct_expressed,
                'n_cells': n_cells,
            })
    
    return pd.DataFrame(results)

# Dictionary to store all dotplot data
dotplot_data_dict = {}

# 1. Core Myeloid
print("\n1. Core Myeloid...")
core_markers = (
    SUBTYPE_MARKERS['Pan_Myeloid'] +
    SUBTYPE_MARKERS['APC_Core'] +
    SUBTYPE_MARKERS['Complement']
)
df_core = calculate_dotplot_stats(adata, core_markers, CELLTYPE_L3_REFINED_COL, 
                                   use_raw=use_raw_for_viz, layer=viz_layer)
if df_core is not None:
    df_core['category'] = 'Core_Myeloid'
    dotplot_data_dict['core_myeloid'] = df_core
    df_core.to_csv(OUTPUT_DIR / 'dotplot_data_core_myeloid.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_core_myeloid.csv ({len(df_core)} rows)")

# 2. Alveolar Macrophages
print("\n2. Alveolar Macrophages...")
am_markers = (
    SUBTYPE_MARKERS['AM_General'] +
    SUBTYPE_MARKERS['AM_Lipid_Handling'] +
    SUBTYPE_MARKERS['AM_Surfactant'] +
    SUBTYPE_MARKERS['AM_Resting']
)
df_am = calculate_dotplot_stats(adata, am_markers, CELLTYPE_L3_REFINED_COL,
                                 use_raw=use_raw_for_viz, layer=viz_layer)
if df_am is not None:
    df_am['category'] = 'Alveolar_Macrophages'
    dotplot_data_dict['alveolar_macrophages'] = df_am
    df_am.to_csv(OUTPUT_DIR / 'dotplot_data_alveolar_macrophages.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_alveolar_macrophages.csv ({len(df_am)} rows)")

# 3. Monocytes & Neutrophils
print("\n3. Monocytes & Neutrophils...")
mono_neut_markers = (
    SUBTYPE_MARKERS['Classical_Mono'] +
    SUBTYPE_MARKERS['Inflammatory_Mono'] +
    SUBTYPE_MARKERS['Non_Classical_Mono'] +
    SUBTYPE_MARKERS['Neutrophil_Core'] +
    SUBTYPE_MARKERS['Neutrophil_Tissue_Stable']
)
df_mono_neut = calculate_dotplot_stats(adata, mono_neut_markers, CELLTYPE_L3_REFINED_COL,
                                        use_raw=use_raw_for_viz, layer=viz_layer)
if df_mono_neut is not None:
    df_mono_neut['category'] = 'Monocytes_Neutrophils'
    dotplot_data_dict['monocytes_neutrophils'] = df_mono_neut
    df_mono_neut.to_csv(OUTPUT_DIR / 'dotplot_data_monocytes_neutrophils.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_monocytes_neutrophils.csv ({len(df_mono_neut)} rows)")

# 4. Dendritic Cells
print("\n4. Dendritic Cells...")
dc_markers = (
    SUBTYPE_MARKERS['cDC2_General'] +
    SUBTYPE_MARKERS['cDC2_Mature'] +
    SUBTYPE_MARKERS['cDC1'] +
    SUBTYPE_MARKERS['pDC_Core'] +
    SUBTYPE_MARKERS['pDC_IFN_Activated'] +
    SUBTYPE_MARKERS['Langerhans_DC'] +
    SUBTYPE_MARKERS['Mature_DC_Migration']
)
df_dc = calculate_dotplot_stats(adata, dc_markers, CELLTYPE_L3_REFINED_COL,
                                 use_raw=use_raw_for_viz, layer=viz_layer)
if df_dc is not None:
    df_dc['category'] = 'Dendritic_Cells'
    dotplot_data_dict['dendritic_cells'] = df_dc
    df_dc.to_csv(OUTPUT_DIR / 'dotplot_data_dendritic_cells.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_dendritic_cells.csv ({len(df_dc)} rows)")

# 5. Interstitial Macrophages
print("\n5. Interstitial Macrophages...")
mac_markers = (
    SUBTYPE_MARKERS['Mac_Inflammatory'] +
    SUBTYPE_MARKERS['Mac_M2_Interstitial'] +
    SUBTYPE_MARKERS['Mac_Resident_IM'] +
    SUBTYPE_MARKERS['Mac_Tissue_Remodeling'] +
    SUBTYPE_MARKERS['Mac_Immunoregulatory'] +
    SUBTYPE_MARKERS['Intestinal_Mac_Niche'] +  # ⭐ 新增
    SUBTYPE_MARKERS['Intestinal_Mac_Stromal_Like'] +  # ⭐ 新增
    SUBTYPE_MARKERS['Intestinal_Mac_Core']  # ⭐ 新增
)
df_mac = calculate_dotplot_stats(adata, mac_markers, CELLTYPE_L3_REFINED_COL,
                                  use_raw=use_raw_for_viz, layer=viz_layer)
if df_mac is not None:
    df_mac['category'] = 'Interstitial_Macrophages'
    dotplot_data_dict['interstitial_macrophages'] = df_mac
    df_mac.to_csv(OUTPUT_DIR / 'dotplot_data_interstitial_macrophages.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_interstitial_macrophages.csv ({len(df_mac)} rows)")

# 6. Mast Cells
print("\n6. Mast Cells...")
df_mast = calculate_dotplot_stats(adata, SUBTYPE_MARKERS['Mast'], CELLTYPE_L3_REFINED_COL,
                                   use_raw=use_raw_for_viz, layer=viz_layer)
if df_mast is not None:
    df_mast['category'] = 'Mast_Cells'
    dotplot_data_dict['mast_cells'] = df_mast
    df_mast.to_csv(OUTPUT_DIR / 'dotplot_data_mast_cells.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_mast_cells.csv ({len(df_mast)} rows)")

# 7. Functional States
print("\n7. Functional States...")
state_markers = []
for m in STATE_MARKERS.values():
    state_markers.extend(m)
df_states = calculate_dotplot_stats(adata, state_markers, CELLTYPE_L3_REFINED_COL,
                                     use_raw=use_raw_for_viz, layer=viz_layer)
if df_states is not None:
    df_states['category'] = 'Functional_States'
    dotplot_data_dict['functional_states'] = df_states
    df_states.to_csv(OUTPUT_DIR / 'dotplot_data_functional_states.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_functional_states.csv ({len(df_states)} rows)")

# 8. Contamination
print("\n8. Contamination Markers...")
contam_markers = []
for m in CONTAMINATION_MARKERS.values():
    contam_markers.extend(m)
df_contam = calculate_dotplot_stats(adata, contam_markers, CELLTYPE_L3_REFINED_COL,
                                     use_raw=use_raw_for_viz, layer=viz_layer)
if df_contam is not None:
    df_contam['category'] = 'Contamination'
    dotplot_data_dict['contamination'] = df_contam
    df_contam.to_csv(OUTPUT_DIR / 'dotplot_data_contamination.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_contamination.csv ({len(df_contam)} rows)")

# 9. Combined long-format file
print("\n9. Creating combined long-format file...")
if dotplot_data_dict:
    df_combined = pd.concat(dotplot_data_dict.values(), ignore_index=True)
    df_combined = df_combined.sort_values(['category', 'cluster', 'gene'])
    df_combined.to_csv(OUTPUT_DIR / 'dotplot_data_ALL_long_format.csv', index=False)
    print(f"  ✓ Saved: dotplot_data_ALL_long_format.csv ({len(df_combined)} rows)")
    
    # Summary statistics
    print(f"\n  Summary:")
    print(f"    Total categories: {df_combined['category'].nunique()}")
    print(f"    Total clusters: {df_combined['cluster'].nunique()}")
    print(f"    Total unique genes: {df_combined['gene'].nunique()}")
    print(f"    Total data points: {len(df_combined):,}")

print("\n" + "="*80)
print("✓ Dotplot data extraction complete")
print("="*80)

In [ ]:
# ===== Convert to Wide Format (Fixed) =====
print("\n" + "="*80)
print("CONVERTING TO WIDE FORMAT")
print("="*80)

def pivot_to_wide_format(df_long, value_col='mean_expression'):
    """
    Convert long format to wide format with duplicate handling
    Rows: genes, Columns: clusters
    """
    if df_long is None or len(df_long) == 0:
        return None
    
    # Check for duplicates and aggregate if needed
    duplicates = df_long.duplicated(subset=['gene', 'cluster'], keep=False)
    if duplicates.any():
        print(f"    ⚠️  Found {duplicates.sum()} duplicate gene-cluster pairs, aggregating...")
        # Use pivot_table with mean aggregation
        df_wide = df_long.pivot_table(
            index='gene',
            columns='cluster',
            values=value_col,
            aggfunc='mean'  # Average if duplicates exist
        )
    else:
        # Use regular pivot if no duplicates
        df_wide = df_long.pivot(
            index='gene',
            columns='cluster',
            values=value_col
        )
    
    return df_wide

# Generate wide format for each category (mean expression)
print("\nGenerating wide format (mean expression)...")
for key, df_long in dotplot_data_dict.items():
    if df_long is not None and len(df_long) > 0:
        # Mean expression
        df_wide_mean = pivot_to_wide_format(df_long, 'mean_expression')
        if df_wide_mean is not None:
            filename = f'dotplot_data_{key}_wide_mean_expression.csv'
            df_wide_mean.to_csv(OUTPUT_DIR / filename)
            print(f"  ✓ {filename} ({df_wide_mean.shape[0]} genes × {df_wide_mean.shape[1]} clusters)")

print("\nGenerating wide format (percent expressed)...")
for key, df_long in dotplot_data_dict.items():
    if df_long is not None and len(df_long) > 0:
        # Percent expressed
        df_wide_pct = pivot_to_wide_format(df_long, 'pct_expressed')
        if df_wide_pct is not None:
            filename = f'dotplot_data_{key}_wide_pct_expressed.csv'
            df_wide_pct.to_csv(OUTPUT_DIR / filename)
            print(f"  ✓ {filename} ({df_wide_pct.shape[0]} genes × {df_wide_pct.shape[1]} clusters)")

# Combined wide format (all categories)
print("\nGenerating combined wide format files...")
if 'df_combined' in locals() and df_combined is not None:
    # Check for duplicates in combined data
    duplicates = df_combined.duplicated(subset=['gene', 'cluster'], keep=False)
    if duplicates.any():
        print(f"  ⚠️  Found {duplicates.sum()} duplicate entries in combined data")
        print(f"      Aggregating duplicates...")
    
    # Mean expression - all genes × all clusters
    df_combined_wide_mean = df_combined.pivot_table(
        index='gene',
        columns='cluster',
        values='mean_expression',
        aggfunc='mean'
    )
    df_combined_wide_mean.to_csv(OUTPUT_DIR / 'dotplot_data_ALL_wide_mean_expression.csv')
    print(f"  ✓ dotplot_data_ALL_wide_mean_expression.csv")
    print(f"    Shape: {df_combined_wide_mean.shape[0]} genes × {df_combined_wide_mean.shape[1]} clusters")
    
    # Percent expressed - all genes × all clusters
    df_combined_wide_pct = df_combined.pivot_table(
        index='gene',
        columns='cluster',
        values='pct_expressed',
        aggfunc='mean'
    )
    df_combined_wide_pct.to_csv(OUTPUT_DIR / 'dotplot_data_ALL_wide_pct_expressed.csv')
    print(f"  ✓ dotplot_data_ALL_wide_pct_expressed.csv")
    print(f"    Shape: {df_combined_wide_pct.shape[0]} genes × {df_combined_wide_pct.shape[1]} clusters")
    
    # Optional: Show which genes have duplicates
    dup_genes = df_combined[duplicates]['gene'].unique()
    if len(dup_genes) > 0:
        print(f"\n  ℹ️  Genes appearing in multiple marker categories ({len(dup_genes)}):")
        print(f"      {', '.join(list(dup_genes)[:20])}")
        if len(dup_genes) > 20:
            print(f"      ... and {len(dup_genes) - 20} more")
    
    # Multi-level format (cluster × metrics)
    print("\nGenerating multi-level wide format...")
    try:
        # Group by gene-cluster and aggregate
        df_agg = df_combined.groupby(['gene', 'cluster']).agg({
            'mean_expression': 'mean',
            'pct_expressed': 'mean',
            'n_cells': 'first'  # n_cells should be same for each cluster
        }).reset_index()
        
        # Create multi-index columns
        df_multi = df_agg.pivot(index='gene', columns='cluster')
        df_multi.columns = ['_'.join(col).strip() for col in df_multi.columns.values]
        df_multi.to_csv(OUTPUT_DIR / 'dotplot_data_ALL_wide_multilevel.csv')
        print(f"  ✓ dotplot_data_ALL_wide_multilevel.csv")
        print(f"    Columns: clusters × [mean_expression, pct_expressed, n_cells]")
    except Exception as e:
        print(f"  ⚠️  Could not create multi-level format: {e}")

print("\n" + "="*80)
print("✓ Wide format conversion complete")
print("="*80)
print("\nGenerated wide format files:")
print("  Per-category (16 files):")
print("    - dotplot_data_*_wide_mean_expression.csv")
print("    - dotplot_data_*_wide_pct_expressed.csv")
print("  Combined (2-3 files):")
print("    - dotplot_data_ALL_wide_mean_expression.csv")
print("    - dotplot_data_ALL_wide_pct_expressed.csv")
print("    - dotplot_data_ALL_wide_multilevel.csv (if successful)")
print("="*80)

In [ ]:
# ===== Verify Generated CSV Files =====
print("\n" + "="*80)
print("VERIFICATION: Generated CSV Files")
print("="*80)

import os

# List all CSV files in output directory
csv_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.csv')])

print(f"\nTotal CSV files generated: {len(csv_files)}")
print("\n" + "-"*80)

# Group by type
long_format = [f for f in csv_files if 'long_format' in f]
wide_mean = [f for f in csv_files if 'wide_mean_expression' in f]
wide_pct = [f for f in csv_files if 'wide_pct_expressed' in f]
wide_multi = [f for f in csv_files if 'wide_multilevel' in f]
other_csv = [f for f in csv_files if f not in long_format + wide_mean + wide_pct + wide_multi]

print("\n📁 LONG FORMAT FILES:")
for f in long_format:
    fpath = OUTPUT_DIR / f
    size_mb = fpath.stat().st_size / 1024 / 1024
    print(f"  ✓ {f} ({size_mb:.2f} MB)")

print("\n📁 WIDE FORMAT (Mean Expression):")
for f in wide_mean:
    fpath = OUTPUT_DIR / f
    size_kb = fpath.stat().st_size / 1024
    print(f"  ✓ {f} ({size_kb:.1f} KB)")

print("\n📁 WIDE FORMAT (Percent Expressed):")
for f in wide_pct:
    fpath = OUTPUT_DIR / f
    size_kb = fpath.stat().st_size / 1024
    print(f"  ✓ {f} ({size_kb:.1f} KB)")

print("\n📁 WIDE FORMAT (Multi-level):")
for f in wide_multi:
    fpath = OUTPUT_DIR / f
    size_mb = fpath.stat().st_size / 1024 / 1024
    print(f"  ✓ {f} ({size_mb:.2f} MB)")

if other_csv:
    print("\n📁 OTHER CSV FILES:")
    for f in other_csv:
        fpath = OUTPUT_DIR / f
        size_kb = fpath.stat().st_size / 1024
        print(f"  ✓ {f} ({size_kb:.1f} KB)")

print("\n" + "="*80)

# Show preview of key files
print("\n📊 PREVIEW: Long Format (first 10 rows)")
print("-"*80)
if long_format:
    df_preview = pd.read_csv(OUTPUT_DIR / long_format[0], nrows=10)
    print(df_preview.to_string(index=False))
    print(f"\n  Total rows in file: {pd.read_csv(OUTPUT_DIR / long_format[0]).shape[0]:,}")

print("\n📊 PREVIEW: Wide Format Mean Expression (first 5 rows × 5 columns)")
print("-"*80)
if wide_mean:
    df_preview = pd.read_csv(OUTPUT_DIR / wide_mean[0], index_col=0, nrows=5)
    print(df_preview.iloc[:, :5].to_string())
    full_shape = pd.read_csv(OUTPUT_DIR / wide_mean[0], index_col=0).shape
    print(f"\n  Full dimensions: {full_shape[0]} genes × {full_shape[1]} clusters")

print("\n📊 PREVIEW: Combined Wide Format (first 5 genes)")
print("-"*80)
all_wide_mean = [f for f in wide_mean if 'ALL_wide_mean' in f]
if all_wide_mean:
    df_all = pd.read_csv(OUTPUT_DIR / all_wide_mean[0], index_col=0, nrows=5)
    print(f"Columns (showing first 8): {list(df_all.columns[:8])}")
    print(f"\n{df_all.iloc[:, :8].to_string()}")
    full_shape = pd.read_csv(OUTPUT_DIR / all_wide_mean[0], index_col=0).shape
    print(f"\n  Full dimensions: {full_shape[0]} genes × {full_shape[1]} clusters")

print("\n" + "="*80)
print("✅ All CSV files successfully generated and verified!")
print(f"📂 Location: {OUTPUT_DIR}")
print("="*80)

## 10. Signature Score Heatmaps

In [ ]:
# ===== Signature Heatmaps =====
if len(score_cols) > 0:
    print("Generating signature score heatmaps...")
    
    score_mean = adata.obs.groupby(CELLTYPE_L3_REFINED_COL)[score_cols].mean()
    score_mean.to_csv(OUTPUT_DIR / 'signature_scores_mean.csv')
    
    # Global heatmap
    display_names = [c.replace('score_', '') for c in score_mean.columns]
    score_mean_display = score_mean.copy()
    score_mean_display.columns = display_names
    
    fig_h = max(10, 0.4 * score_mean.shape[0] + 2)
    fig_w = max(14, 0.3 * score_mean.shape[1] + 4)
    
    plt.figure(figsize=(fig_w, fig_h))
    sns.heatmap(score_mean_display, cmap='RdBu_r', center=0,
                cbar_kws={'label': 'Mean Score'},
                xticklabels=True, yticklabels=True,
                linewidths=0.5, linecolor='lightgray')
    plt.title('Signature Scores by Refined Cell Type', fontsize=14, pad=20)
    plt.xlabel('Signature', fontsize=12)
    plt.ylabel('Cell Type', fontsize=12)
    plt.xticks(rotation=90, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'heatmap_signatures_global.png', dpi=300, bbox_inches='tight')
    plt.savefig(OUTPUT_DIR / 'heatmap_signatures_global.pdf', bbox_inches='tight')
    plt.show()
    
    print("  ✓ Saved: heatmap_signatures_global (PNG + PDF)")
    
    # Category-specific
    for prefix, title in [('SUB', 'Subtype'), ('STATE', 'State'), ('CONTAM', 'Contamination')]:
        cat_cols = [c for c in score_cols if c.startswith(f'score_{prefix}_')]
        if len(cat_cols) == 0:
            continue
        
        cat_data = score_mean[cat_cols].copy()
        cat_data.columns = [c.replace(f'score_{prefix}_', '') for c in cat_cols]
        
        fig_h = max(8, 0.4 * cat_data.shape[0] + 2)
        fig_w = max(10, 0.5 * cat_data.shape[1] + 3)
        
        plt.figure(figsize=(fig_w, fig_h))
        sns.heatmap(cat_data, cmap='RdBu_r', center=0,
                    cbar_kws={'label': 'Mean Score'},
                    xticklabels=True, yticklabels=True,
                    linewidths=0.5, linecolor='lightgray')
        plt.title(f'{title} Signatures', fontsize=14, pad=20)
        plt.xlabel('Signature', fontsize=12)
        plt.ylabel('Cell Type', fontsize=12)
        plt.xticks(rotation=45, ha='right', fontsize=10)
        plt.yticks(rotation=0, fontsize=9)
        plt.tight_layout()
        
        fname = f'heatmap_signatures_{prefix.lower()}'
        plt.savefig(OUTPUT_DIR / f'{fname}.png', dpi=300, bbox_inches='tight')
        plt.savefig(OUTPUT_DIR / f'{fname}.pdf', bbox_inches='tight')
        plt.close()
        
        print(f"  ✓ Saved: {fname} ({cat_data.shape[1]} signatures)")
    
    print("\n✓ Signature heatmaps complete")

## 11. AM Doublet Analysis (P1-4)

Quantitative assessment of AM vs AT2 program enrichment

In [ ]:
# ===== P1-4: AM Doublet Analysis =====
if len(score_cols) > 0:
    print("\nAM Doublet Analysis (P1-4)...")
    print("=" * 80)
    
    # Extract relevant scores
    am_score_col = 'score_SUB_AM_General'
    at2_score_col = 'score_CONTAM_AT2_Alveolar'
    
    if am_score_col in score_cols and at2_score_col in score_cols:
        am_clusters = adata.obs[
            adata.obs[CELLTYPE_L3_REFINED_COL].str.contains('AM|Alveolar', case=False, na=False)
        ]
        
        am_doublet_analysis = am_clusters.groupby(CELLTYPE_L3_REFINED_COL).agg({
            am_score_col: 'mean',
            at2_score_col: 'mean',
            CELLTYPE_L3_REFINED_COL: 'size'
        }).rename(columns={CELLTYPE_L3_REFINED_COL: 'n_cells'})
        
        am_doublet_analysis['AM_AT2_ratio'] = (
            am_doublet_analysis[am_score_col] / 
            (am_doublet_analysis[at2_score_col] + 1e-6)
        )
        am_doublet_analysis['doublet_risk'] = 'Low'
        am_doublet_analysis.loc[
            (am_doublet_analysis[at2_score_col] > 0.5) &
            (am_doublet_analysis['AM_AT2_ratio'] < 2),
            'doublet_risk'
        ] = 'High'
        
        am_doublet_analysis = am_doublet_analysis.sort_values('AM_AT2_ratio')
        am_doublet_analysis.to_csv(OUTPUT_DIR / 'am_doublet_analysis.csv')
        
        print("\nAM vs AT2 Program Enrichment:")
        print("-" * 80)
        for ct, row in am_doublet_analysis.iterrows():
            risk_flag = " ⚠️  HIGH DOUBLET RISK" if row['doublet_risk'] == 'High' else ""
            print(f"\n  {ct} ({row['n_cells']:.0f} cells){risk_flag}")
            print(f"    AM program: {row[am_score_col]:.3f}")
            print(f"    AT2 program: {row[at2_score_col]:.3f}")
            print(f"    AM/AT2 ratio: {row['AM_AT2_ratio']:.2f}")
        
        print("\n" + "=" * 80)
        print("  ✓ Saved: am_doublet_analysis.csv")
    else:
        print("  ⚠️  Required scores not found, skipping AM doublet analysis")

In [ ]:
# ===== UPDATED Cell Type Re-annotation (Final Version) =====
print("\n" + "="*80)
print("UPDATING CELL TYPE ANNOTATIONS - FINAL VERSION")
print("="*80)

CELLTYPE_REANNOTATION_FINAL = {
    # Alveolar macrophages
    'Alveolar macrophages_c0': 'Resident Alveolar macrophages',
    'Alveolar macrophages_c1': 'Resident Alveolar macrophages',
    'Alveolar macrophages_c2': 'Resident Alveolar macrophages',
    'Alveolar macrophages_c3': 'Resting Alveolar macrophages',
    
    # Neutrophils / Monocytes
    'Classical monocytes_c0': 'Neutrophils',  # L2 corrected
    'Classical monocytes_c1': 'Typical Classical monocytes',
    'Classical monocytes_c2': 'Inflammatory Classical monocytes',
    
    # cDC2 (unified from DC2 and DC)
    'DC2_c0': 'Conventional cDC2',
    'DC2_c1': 'Langerhans-like cDC2',
    'DC_c0': 'Conventional cDC2',
    'DC_c1': 'Antigen-presenting cDC2',
    
    # pDC (common term exception)
    'pDC_c0': 'pDC',
    'pDC_c1': 'pDC',
    
    # Interstitial macrophages (from Macrophages)
    'Macrophages_c0': 'Inflammatory Interstitial macrophages',
    'Macrophages_c1': 'M2-like Interstitial macrophages',
    'Macrophages_c2': 'Atypically activated Interstitial macrophages',
    
    # Mast cells (common term exception)
    'Mast cells_c0': 'Mast cells',
    'Mast cells_c1': 'Mast cells',
    
    # Interstitial macrophages (from Intestinal macrophages)
    'Intestinal macrophages_c0': 'CD163L1+ Interstitial macrophages',
    'Intestinal macrophages_c1': 'Low-quality Interstitial macrophages',
    'Intestinal macrophages_c2': 'Tissue-remodeling Interstitial macrophages',
    'Intestinal macrophages_c3': 'Immunoregulatory Interstitial macrophages',
    'Intestinal macrophages_c4': 'Stromal-like Interstitial macrophages',
}

LOW_CONFIDENCE_CLUSTERS_FINAL = [
    'Macrophages_c2',  # Atypically activated (state-like)
    'Intestinal macrophages_c1',  # Low-quality
    'Intestinal macrophages_c3',  # Immunoregulatory (low-confidence)
    'Intestinal macrophages_c4',  # Stromal-like (low-confidence)
]

# Apply final annotation
CELLTYPE_L3_FINAL_COL = 'cell_type_L3_final'
adata.obs[CELLTYPE_L3_FINAL_COL] = adata.obs[CELLTYPE_L3_COL].map(CELLTYPE_REANNOTATION_FINAL)

# Handle unmapped
unmapped = adata.obs[CELLTYPE_L3_COL][adata.obs[CELLTYPE_L3_FINAL_COL].isna()].unique()
if len(unmapped) > 0:
    print(f"⚠️  {len(unmapped)} unmapped clusters - keeping original labels")
    adata.obs[CELLTYPE_L3_FINAL_COL].fillna(adata.obs[CELLTYPE_L3_COL], inplace=True)

adata.obs[CELLTYPE_L3_FINAL_COL] = adata.obs[CELLTYPE_L3_FINAL_COL].astype('category')

print(f"\n✓ Final annotation applied: {adata.obs[CELLTYPE_L3_FINAL_COL].nunique()} unique cell types")

# Display
print("\n" + "="*80)
print("FINAL CELL TYPE DISTRIBUTION")
print("="*80)
final_counts = adata.obs[CELLTYPE_L3_FINAL_COL].value_counts()
for ct, count in final_counts.items():
    pct = count / adata.n_obs * 100
    original = adata.obs[adata.obs[CELLTYPE_L3_FINAL_COL] == ct][CELLTYPE_L3_COL].iloc[0]
    flag = " [⚠️ LOW CONFIDENCE]" if original in LOW_CONFIDENCE_CLUSTERS_FINAL else ""
    print(f"  {ct}: {count:,} cells ({pct:.1f}%){flag}")
print("="*80)

In [ ]:
# ===== UPDATED Marker Gene Sets (Final Version) =====
print("\n" + "="*80)
print("DEFINING FINAL MARKER GENE SETS")
print("="*80)

VALIDATION_MARKERS_FINAL = {
    # Alveolar macrophages
    'Resident_Alveolar_Mac': ['MARCO', 'FABP4', 'PPARG', 'APOE', 'MSR1', 'MERTK', 'SIGLEC1', 'CHIT1'],
    'Resting_Alveolar_Mac': ['MARCO', 'FABP4', 'PPARG', 'MRC1', 'MERTK', 'IL10', 'TGFB1', 'MSR1'],
    
    # Neutrophils
    'Neutrophils': ['S100A8', 'S100A9', 'FCGR3B', 'CSF3R', 'CXCR2', 'LCN2', 'MPO', 'ELANE'],
    
    # Classical monocytes
    'Typical_Classical_Mono': ['FCN1', 'LST1', 'S100A8', 'S100A9', 'CTSS', 'LGALS3', 'LILRB1', 'TYMP'],
    'Inflammatory_Classical_Mono': ['IL1B', 'TNF', 'CXCL8', 'NFKBIA', 'PTX3', 'CCL2', 'CCL3', 'CCL20'],
    
    # cDC2
    'Conventional_cDC2': ['CD1C', 'FCER1A', 'CLEC10A', 'CD1E', 'CST3', 'HLA-DRA', 'IRF4', 'ITGAX'],
    'Langerhans_like_cDC2': ['CD207', 'CD1A', 'FCER1A', 'CLEC10A', 'CCR6', 'HLA-DRA', 'CST3'],  # EPCAM removed
    'Antigen_presenting_cDC2': ['LAMP3', 'CCR7', 'FSCN1', 'CD74', 'HLA-DRA', 'HLA-DPB1', 'HLA-DPA1', 'MARCKSL1'],
    
    # pDC
    'pDC': ['GZMB', 'CLEC4C', 'IL3RA', 'TCF4', 'IRF7', 'SPIB', 'SERPINF1', 'PTCRA'],
    
    # Mast cells
    'Mast_cells': ['TPSAB1', 'TPSB2', 'KIT', 'CPA3', 'MS4A2', 'HDC', 'GATA2', 'LTC4S'],
    
    # Interstitial macrophages (comprehensive)
    'Inflammatory_Interstitial_Mac': ['IL1B', 'PTX3', 'CCL20', 'TNF', 'NFKBIA', 'TIMP1', 'VEGFA', 'IL10'],
    'M2_like_Interstitial_Mac': ['MRC1', 'CD163', 'FOLR2', 'STAB1', 'MERTK', 'C1QA', 'C1QB', 'C1QC'],
    'Atypically_activated_Interstitial_Mac': ['CXCL8', 'NFKBIA', 'MARCKS', 'VEGFA', 'SPP1', 'TIMP1', 'IL1B', 'CCL20'],
    'CD163L1_Interstitial_Mac': ['CD163L1', 'SELENOP', 'FOLR2', 'LYVE1', 'MRC1', 'CD163', 'C1QC', 'STAB1'],
    'Low_quality_Interstitial_Mac': ['LST1', 'TYROBP', 'FCER1G', 'CTSS', 'IL1B', 'MS4A7', 'C1QC', 'MARCKS'],
    'Tissue_remodeling_Interstitial_Mac': ['F13A1', 'STAB1', 'MRC1', 'LYVE1', 'GAS6', 'MERTK', 'TGFB1', 'MMP12'],
    'Immunoregulatory_Interstitial_Mac': ['IL10', 'TGFB1', 'MERTK', 'MRC1', 'CD163', 'FOLR2', 'C1QC', 'STAB1'],
    'Stromal_like_Interstitial_Mac': ['CXCL12', 'PLXDC1', 'DCN', 'LUM', 'COL1A1', 'C1QC', 'MRC1', 'CD163'],
}

# Create ordered marker list for dotplot (grouped by cell type)
DOTPLOT_MARKERS_ORDERED = []
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Resident_Alveolar_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Resting_Alveolar_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Neutrophils'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Typical_Classical_Mono'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Inflammatory_Classical_Mono'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Conventional_cDC2'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Langerhans_like_cDC2'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Antigen_presenting_cDC2'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['pDC'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Mast_cells'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Inflammatory_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['M2_like_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Atypically_activated_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['CD163L1_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Low_quality_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Tissue_remodeling_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Immunoregulatory_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED.extend(VALIDATION_MARKERS_FINAL['Stromal_like_Interstitial_Mac'])

# Remove duplicates while preserving order
seen = set()
DOTPLOT_MARKERS_UNIQUE = []
for marker in DOTPLOT_MARKERS_ORDERED:
    if marker not in seen:
        DOTPLOT_MARKERS_UNIQUE.append(marker)
        seen.add(marker)

print(f"\n✓ Defined {len(VALIDATION_MARKERS_FINAL)} marker sets")
print(f"✓ Total unique markers for validation: {len(DOTPLOT_MARKERS_UNIQUE)}")
print("\nMarker categories:")
for category, markers in VALIDATION_MARKERS_FINAL.items():
    print(f"  {category}: {len(markers)} genes")

In [ ]:
# ===== COMPREHENSIVE VALIDATION DOTPLOT =====
print("\n" + "="*80)
print("GENERATING COMPREHENSIVE VALIDATION DOTPLOT")
print("="*80)

# Check marker availability
gene_names_viz = adata.raw.var_names if use_raw_for_viz else adata.var_names
available_markers = [m for m in DOTPLOT_MARKERS_UNIQUE if m in gene_names_viz]
missing_markers = [m for m in DOTPLOT_MARKERS_UNIQUE if m not in gene_names_viz]

print(f"\nMarker availability:")
print(f"  Available: {len(available_markers)}/{len(DOTPLOT_MARKERS_UNIQUE)} ({len(available_markers)/len(DOTPLOT_MARKERS_UNIQUE)*100:.1f}%)")
if missing_markers:
    print(f"  Missing: {len(missing_markers)} genes")
    print(f"    {', '.join(missing_markers[:10])}")
    if len(missing_markers) > 10:
        print(f"    ... and {len(missing_markers)-10} more")

print(f"\nGenerating dotplot with {len(available_markers)} markers × {adata.obs[CELLTYPE_L3_FINAL_COL].nunique()} cell types...")

# Create custom ordering for cell types (match annotation table)
celltype_order = [
    'Resident Alveolar macrophages',
    'Resting Alveolar macrophages',
    'Neutrophils',
    'Typical Classical monocytes',
    'Inflammatory Classical monocytes',
    'Conventional cDC2',
    'Langerhans-like cDC2',
    'Antigen-presenting cDC2',
    'pDC',
    'Mast cells',
    'Inflammatory Interstitial macrophages',
    'M2-like Interstitial macrophages',
    'Atypically activated Interstitial macrophages',
    'CD163L1+ Interstitial macrophages',
    'Low-quality Interstitial macrophages',
    'Tissue-remodeling Interstitial macrophages',
    'Immunoregulatory Interstitial macrophages',
    'Stromal-like Interstitial macrophages',
]

# Filter to existing cell types
celltype_order_filtered = [ct for ct in celltype_order if ct in adata.obs[CELLTYPE_L3_FINAL_COL].values]
print(f"  Ordered cell types: {len(celltype_order_filtered)}")

# Reorder adata.obs for plotting
adata.obs[CELLTYPE_L3_FINAL_COL] = pd.Categorical(
    adata.obs[CELLTYPE_L3_FINAL_COL],
    categories=celltype_order_filtered,
    ordered=True
)

# Generate dotplot
try:
    print("\nAttempting to generate dotplot...")
    
    # Calculate appropriate figure size
    n_genes = len(available_markers)
    n_celltypes = len(celltype_order_filtered)
    fig_width = max(20, n_genes * 0.3 + 4)
    fig_height = max(12, n_celltypes * 0.5 + 2)
    
    print(f"  Figure size: {fig_width:.1f} × {fig_height:.1f} inches")
    
    # Try modern scanpy API
    try:
        dp = sc.pl.dotplot(
            adata,
            var_names=available_markers,
            groupby=CELLTYPE_L3_FINAL_COL,
            use_raw=use_raw_for_viz,
            layer=None if use_raw_for_viz else viz_layer,
            standard_scale='var',
            show=False,
            return_fig=True,
            figsize=(fig_width, fig_height),
        )
        
        if dp is not None and hasattr(dp, 'fig'):
            dp.fig.suptitle('Comprehensive Myeloid Cell Type Validation', 
                           fontsize=16, y=0.995, weight='bold')
            
            # Save
            dp.fig.savefig(OUTPUT_DIR / 'dotplot_COMPREHENSIVE_VALIDATION.png', 
                          dpi=300, bbox_inches='tight')
            dp.fig.savefig(OUTPUT_DIR / 'dotplot_COMPREHENSIVE_VALIDATION.pdf', 
                          bbox_inches='tight')
            plt.close(dp.fig)
            
            print(f"\n  ✓ Saved: dotplot_COMPREHENSIVE_VALIDATION (PNG + PDF)")
            print(f"    Dimensions: {n_genes} markers × {n_celltypes} cell types")
        else:
            raise TypeError("DotPlot object is None")
            
    except (TypeError, AttributeError) as e:
        print(f"  ⚠️  Modern API failed: {e}")
        print(f"  Trying fallback method...")
        
        # Fallback: Create figure and plot
        fig = plt.figure(figsize=(fig_width, fig_height))
        
        sc.pl.dotplot(
            adata,
            var_names=available_markers,
            groupby=CELLTYPE_L3_FINAL_COL,
            use_raw=use_raw_for_viz,
            layer=None if use_raw_for_viz else viz_layer,
            standard_scale='var',
            show=False,
        )
        
        plt.suptitle('Comprehensive Myeloid Cell Type Validation', 
                    fontsize=16, y=0.995, weight='bold')
        plt.tight_layout()
        
        plt.savefig(OUTPUT_DIR / 'dotplot_COMPREHENSIVE_VALIDATION.png', 
                   dpi=300, bbox_inches='tight')
        plt.savefig(OUTPUT_DIR / 'dotplot_COMPREHENSIVE_VALIDATION.pdf', 
                   bbox_inches='tight')
        plt.close()
        
        print(f"\n  ✓ Saved: dotplot_COMPREHENSIVE_VALIDATION (PNG + PDF)")
        print(f"    Dimensions: {n_genes} markers × {n_celltypes} cell types")
    
    print("\n" + "="*80)
    print("✅ COMPREHENSIVE VALIDATION DOTPLOT COMPLETE")
    print("="*80)
    
    # Summary statistics
    print("\nValidation Summary:")
    print(f"  Total cell types validated: {n_celltypes}")
    print(f"  Total markers used: {n_genes}")
    print(f"  Low-confidence clusters flagged: {len(LOW_CONFIDENCE_CLUSTERS_FINAL)}")
    print(f"\n  Key features to check:")
    print(f"    - Neutrophils: Should show S100A8/S100A9/FCGR3B high")
    print(f"    - AM clusters: MARCO/FABP4/PPARG should be high")
    print(f"    - cDC2 subtypes: Check CD1C/FCER1A vs LAMP3/CCR7 patterns")
    print(f"    - Interstitial Mac diversity: Check CD163L1/SELENOP vs inflammatory markers")
    print(f"    - Stromal-like: Should show DCN/LUM/CXCL12 (contamination concern)")
    
except Exception as e:
    print(f"\n❌ Failed to generate comprehensive dotplot: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*80)

In [ ]:
# ===== UPDATED Marker Gene Sets (Refined Version - Core Axes) =====
print("\n" + "="*80)
print("DEFINING REFINED MARKER GENE SETS - CORE BIOLOGICAL AXES")
print("="*80)

VALIDATION_MARKERS_REFINED = {
    # Alveolar macrophages (clear axis separation)
    'Resident_Alveolar_Mac': ['MARCO', 'FABP4', 'PPARG', 'SIGLEC1', 'CHIT1', 'MSR1', 'SLC40A1', 'APOE'],
    'Resting_Alveolar_Mac': ['IL10', 'TGFB1', 'MERTK', 'MRC1', 'MSR1', 'APOE', 'GAS6', 'TREM2'],
    
    # Neutrophils
    'Neutrophils': ['FCGR3B', 'CSF3R', 'CXCR2', 'MPO', 'ELANE', 'LCN2', 'S100A8', 'S100A9'],
    
    # Classical monocytes (distinct axes)
    'Typical_Classical_Mono': ['FCN1', 'S100A8', 'S100A9', 'LILRB1', 'LGALS3', 'CTSS'],
    'Inflammatory_Classical_Mono': ['IL1B', 'CXCL8', 'PTX3', 'NFKBIA', 'TNF', 'CCL20'],
    
    # cDC2 subtypes
    'Conventional_cDC2': ['CD1C', 'FCER1A', 'CD1E', 'CLEC10A', 'IRF4', 'CST3'],
    'Langerhans_like_cDC2': ['CD207', 'CD1A', 'CCR6', 'EPCAM', 'LILRB4', 'CXCL14'],
    'Antigen_presenting_cDC2': ['LAMP3', 'CCR7', 'FSCN1', 'RELB', 'MARCKSL1', 'IL7R'],
    
    # pDC
    'pDC': ['CLEC4C', 'IL3RA', 'GZMB', 'TCF4', 'IRF7', 'SERPINF1'],
    
    # Mast cells
    'Mast_cells': ['TPSB2', 'TPSAB1', 'CPA3', 'MS4A2', 'KIT', 'HDC', 'GATA2'],
    
    # Interstitial macrophages (核心区分轴)
    'Inflammatory_Interstitial_Mac': ['CCL20', 'PTX3', 'TIMP1', 'IL1B', 'VEGFA', 'SPP1'],
    'Atypically_activated_Interstitial_Mac': ['CXCL8', 'NFKBIA', 'TIMP1', 'VEGFA', 'SPP1', 'CCL20'],
    'M2_like_Interstitial_Mac': ['C1QA', 'C1QB', 'C1QC', 'STAB1', 'FOLR2', 'MRC1'],
    'CD163L1_Interstitial_Mac': ['CD163L1', 'SELENOP', 'LYVE1', 'FOLR2', 'ABCA6', 'C1QC'],  # 核心！
    'Tissue_remodeling_Interstitial_Mac': ['F13A1', 'MMP12', 'FN1', 'TIMP1', 'GAS6', 'TGFB1'],
    'Immunoregulatory_Interstitial_Mac': ['IL10', 'TGFB1', 'MERTK', 'FOLR2', 'CD163L1', 'LYVE1'],
    'Stromal_like_Interstitial_Mac': ['CXCL12', 'PLXDC1', 'DCN', 'LUM', 'COL1A1', 'CD163L1'],  # 一眼区分
    'Low_quality_Interstitial_Mac': ['IL1B', 'CXCL8', 'NFKBIA', 'PTX3', 'TNF', 'MARCKS'],
}

# Create ordered marker list for dotplot (grouped by cell type)
DOTPLOT_MARKERS_ORDERED_V2 = []

# Alveolar macrophages
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Resident_Alveolar_Mac'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Resting_Alveolar_Mac'])

# Neutrophils & Monocytes
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Neutrophils'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Typical_Classical_Mono'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Inflammatory_Classical_Mono'])

# DCs
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Conventional_cDC2'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Langerhans_like_cDC2'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Antigen_presenting_cDC2'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['pDC'])

# Mast cells
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Mast_cells'])

# Interstitial macrophages
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Inflammatory_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Atypically_activated_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['M2_like_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['CD163L1_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Tissue_remodeling_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Immunoregulatory_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Stromal_like_Interstitial_Mac'])
DOTPLOT_MARKERS_ORDERED_V2.extend(VALIDATION_MARKERS_REFINED['Low_quality_Interstitial_Mac'])

# Remove duplicates while preserving order
seen = set()
DOTPLOT_MARKERS_UNIQUE_V2 = []
for marker in DOTPLOT_MARKERS_ORDERED_V2:
    if marker not in seen:
        DOTPLOT_MARKERS_UNIQUE_V2.append(marker)
        seen.add(marker)

print(f"\n✓ Defined {len(VALIDATION_MARKERS_REFINED)} marker sets")
print(f"✓ Total unique markers: {len(DOTPLOT_MARKERS_UNIQUE_V2)}")
print("\nMarker categories:")
for category, markers in VALIDATION_MARKERS_REFINED.items():
    print(f"  {category}: {len(markers)} genes")

In [ ]:
# ===== COMPREHENSIVE VALIDATION DOTPLOT V2 (Refined Markers) =====
print("\n" + "="*80)
print("GENERATING COMPREHENSIVE VALIDATION DOTPLOT - V2 REFINED")
print("="*80)

# Check marker availability
gene_names_viz = adata.raw.var_names if use_raw_for_viz else adata.var_names
available_markers_v2 = [m for m in DOTPLOT_MARKERS_UNIQUE_V2 if m in gene_names_viz]
missing_markers_v2 = [m for m in DOTPLOT_MARKERS_UNIQUE_V2 if m not in gene_names_viz]

print(f"\nMarker availability:")
print(f"  Available: {len(available_markers_v2)}/{len(DOTPLOT_MARKERS_UNIQUE_V2)} ({len(available_markers_v2)/len(DOTPLOT_MARKERS_UNIQUE_V2)*100:.1f}%)")
if missing_markers_v2:
    print(f"  Missing ({len(missing_markers_v2)}): {', '.join(missing_markers_v2[:20])}")
    if len(missing_markers_v2) > 20:
        print(f"    ... and {len(missing_markers_v2)-20} more")

# Cell type ordering (match annotation table)
celltype_order_v2 = [
    'Resident Alveolar macrophages',
    'Resting Alveolar macrophages',
    'Neutrophils',
    'Typical Classical monocytes',
    'Inflammatory Classical monocytes',
    'Conventional cDC2',
    'Langerhans-like cDC2',
    'Antigen-presenting cDC2',
    'pDC',
    'Mast cells',
    'Inflammatory Interstitial macrophages',
    'Atypically activated Interstitial macrophages',
    'M2-like Interstitial macrophages',
    'CD163L1+ Interstitial macrophages',
    'Tissue-remodeling Interstitial macrophages',
    'Immunoregulatory Interstitial macrophages',
    'Stromal-like Interstitial macrophages',
    'Low-quality Interstitial macrophages',
]

# Filter to existing cell types
celltype_order_v2_filtered = [ct for ct in celltype_order_v2 if ct in adata.obs[CELLTYPE_L3_FINAL_COL].values]

# Reorder categorical
adata.obs[CELLTYPE_L3_FINAL_COL] = pd.Categorical(
    adata.obs[CELLTYPE_L3_FINAL_COL],
    categories=celltype_order_v2_filtered,
    ordered=True
)

# Calculate figure size
n_genes = len(available_markers_v2)
n_celltypes = len(celltype_order_v2_filtered)
fig_width = max(28, n_genes * 0.25)
fig_height = max(12, n_celltypes * 0.65)

print(f"\nGenerating dotplot:")
print(f"  Markers: {n_genes}")
print(f"  Cell types: {n_celltypes}")
print(f"  Figure size: {fig_width:.1f} × {fig_height:.1f} inches")

# DIRECT METHOD
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

sc.pl.dotplot(
    adata,
    var_names=available_markers_v2,
    groupby=CELLTYPE_L3_FINAL_COL,
    use_raw=use_raw_for_viz,
    layer=None if use_raw_for_viz else viz_layer,
    standard_scale='var',
    show=False,
    ax=ax,
)

# Add title
fig.suptitle('Comprehensive Myeloid Cell Type Validation - Core Biological Axes', 
            fontsize=18, y=0.998, weight='bold')

# Tight layout
plt.tight_layout()

# Save
output_png_v2 = OUTPUT_DIR / 'dotplot_COMPREHENSIVE_VALIDATION_V2_REFINED.png'
output_pdf_v2 = OUTPUT_DIR / 'dotplot_COMPREHENSIVE_VALIDATION_V2_REFINED.pdf'

plt.savefig(output_png_v2, dpi=300, bbox_inches='tight')
plt.savefig(output_pdf_v2, bbox_inches='tight')
plt.close()

# Verify
if output_png_v2.exists() and output_pdf_v2.exists():
    png_size = output_png_v2.stat().st_size / 1024 / 1024
    pdf_size = output_pdf_v2.stat().st_size / 1024 / 1024
    
    print(f"\n✅ FILES GENERATED SUCCESSFULLY:")
    print(f"  📊 PNG: {output_png_v2.name} ({png_size:.2f} MB)")
    print(f"  📊 PDF: {output_pdf_v2.name} ({pdf_size:.2f} MB)")
    
    print(f"\n📋 VALIDATION CHECKLIST:")
    print(f"  ✓ Resident vs Resting AM: Check MARCO/FABP4/PPARG vs IL10/TGFB1/MERTK")
    print(f"  ✓ Neutrophils: S100A8/S100A9/FCGR3B should be highly specific")
    print(f"  ✓ Typical vs Inflammatory Mono: FCN1/S100A8 vs IL1B/CXCL8/PTX3")
    print(f"  ✓ cDC2 subtypes: CD1C vs CD207/CD1A vs LAMP3/CCR7")
    print(f"  ✓ CD163L1+ IM: CD163L1/SELENOP/LYVE1 - 核心niche marker")
    print(f"  ✓ Stromal-like IM: CXCL12/PLXDC1/DCN/LUM - 一眼区分")
    print(f"  ✓ Inflammatory IM: CCL20/PTX3/TIMP1 vs Atypical (CXCL8/NFKBIA)")
    
else:
    print(f"\n❌ ERROR: Files not created!")
    print(f"  Expected PNG: {output_png_v2}")
    print(f"  Expected PDF: {output_pdf_v2}")

print("\n" + "="*80)
print("✅ COMPREHENSIVE VALIDATION DOTPLOT V2 COMPLETE")
print("="*80)

## 12. Save Refined AnnData & Summary

In [ ]:
# ===== Save Refined AnnData =====
print("\nSaving refined AnnData...")

output_h5ad = OUTPUT_DIR / 'adata_myeloid_refined_optimized.h5ad'

adata.uns['annotation_refinement_info'] = {
    'date': pd.Timestamp.now().isoformat(),
    'original_column': CELLTYPE_L3_COL,
    'refined_column': CELLTYPE_L3_REFINED_COL,
    'n_clusters_original': adata.obs[CELLTYPE_L3_COL].nunique(),
    'n_clusters_refined': adata.obs[CELLTYPE_L3_REFINED_COL].nunique(),
    'low_confidence_clusters': LOW_CONFIDENCE_CLUSTERS,
    'optimizations': [
        'P0-1: Dual marker coverage (var vs raw)',
        'P0-2: Fixed dotplot saving mechanism',
        'P0-3: UMAP existence check',
        'P1-1: pDC_IFN_Activated (no plasma contamination)',
        'P1-2: Neutrophil tissue-stable genes',
        'P1-3: Resident IM marker axis',
        'P1-4: AM doublet quantitative analysis',
        'P2-2: Reproducible signature scoring',
        'P2-4: Vector format outputs (PDF)',
    ]
}

adata.write_h5ad(output_h5ad, compression='gzip', compression_opts=9)
print(f"  ✓ Saved: {output_h5ad.name}")
print(f"  Size: {output_h5ad.stat().st_size / 1024**2:.1f} MB")

In [ ]:
# ===== Final Summary =====
print("\n" + "="*80)
print("ANALYSIS COMPLETE - OPTIMIZED VERSION")
print("="*80)
print()

print("📁 Output Directory:")
print(f"   {OUTPUT_DIR}")
print()

print("📊 Generated Files:")
print("-" * 80)
print()

print("Data & Reports:")
print("  - adata_myeloid_refined_optimized.h5ad")
print("  - marker_coverage_dual_report.csv (P0-1: var vs raw)")
if len(score_cols) > 0:
    print("  - signature_scores_mean.csv")
    if 'am_doublet_analysis' in locals():
        print("  - am_doublet_analysis.csv (P1-4)")
print()

print("Visualizations (PNG + PDF):")
print("  UMAPs:")
print("    - umap_original_vs_refined")
print("    - umap_markers_* (per category)")
print()
print("  Dotplots (P0-2 fixed):")
print("    - dotplot_core_myeloid")
print("    - dotplot_alveolar_macrophages")
print("    - dotplot_monocytes_neutrophils (P1-2 enhanced)")
print("    - dotplot_dendritic_cells (P1-1 pDC fixed)")
print("    - dotplot_interstitial_macrophages (P1-3 enhanced)")
print("    - dotplot_mast_cells")
print("    - dotplot_functional_states")
print("    - dotplot_contamination")
print()
if len(score_cols) > 0:
    print("  Heatmaps:")
    print("    - heatmap_signatures_global")
    print("    - heatmap_signatures_sub/state/contam")
print()

print("="*80)
print()

print("✅ KEY OPTIMIZATIONS APPLIED:")
print("-" * 80)
print("  P0-1: Dual marker coverage reporting (HVG vs full-gene)")
print("  P0-2: Fixed dotplot saving mechanism")
print("  P0-3: UMAP existence validation")
print("  P1-1: pDC_IFN_Activated uses ISG markers (no plasma contamination)")
print("  P1-2: Neutrophil panel includes tissue-stable genes")
print("  P1-3: Added Mac_Resident_IM axis")
print("  P1-4: Quantitative AM doublet analysis")
print("  P2-2: Reproducible signature scoring (fixed random seed)")
print("  P2-4: Vector format outputs (PDF + PNG)")
print()

print("🎯 Next Steps:")
print("-" * 80)
print("  1. Review marker_coverage_dual_report.csv")
print("  2. Check am_doublet_analysis.csv for AT2 contamination")
print("  3. Validate low-confidence clusters")
print("  4. Use refined annotations for downstream analysis")
print()

print("="*80)
print("✅ MYELOID VALIDATION COMPLETE (OPTIMIZED)")
print("="*80)